In [ ]:
import boto3
from sagemaker.session import Session
from sagemaker import get_execution_role,image_uris
# Define a SageMaker estimator and fit it
from sagemaker.estimator import Estimator
from sklearn import datasets
import pandas as pd
from sagemaker.inputs import TrainingInput
from tensorflow.keras.datasets import fashion_mnist

# Profile and Session

In [ ]:

# Your custom profile and role ARN
profile_name = '<AWS_PROFILE>'
sagemaker_role = 'arn:aws:iam::<ACCOUNT_ID>:role/service-role/<SAGEMAKER_EXECUTION_ROLE>'  

# Initialize a boto3 session using the custom profile
boto_session = boto3.Session(profile_name=profile_name)

# Initialize the SageMaker session with the boto3 session
sagemaker_session = Session(boto_session=boto_session)


# S3 Client

In [ ]:
# Use the session to create an S3 client
s3_client = boto_session.client('s3')

# Specify your bucket name and desired key/prefix
bucket_name = '<S3_BUCKET>'
object_key = 'iris_data.csv'

# Upload data to S3

In [ ]:
prefix = "data" # path from bucket to data 

train_name = 'AIDM_timeseries_train.csv' # using same path in the S3 
test_name = 'AIDM_timeseries_train.csv'

local_data_path = 'data'

# Method with more control (since uses client session)
for data_name in [train_name, test_name]:
    s3_client.upload_file(
        Filename=os.path.join(local_data_path, data_name),
        Bucket=bucket_name,
        Key=os.path.join(prefix, data_name)
    )

# Load Model from SageMaker

In [ ]:
# Get the XGBoost image URI for the region
xgboost_image_uri = image_uris.retrieve(framework='xgboost', region=boto_session.region_name, version='latest')
sagemaker_session = Session(boto_session=boto_session)

# Set Model 

In [ ]:
xgboost_estimator = XGBoost(
    image_uri=xgboost_image_uri,
    entry_point='/train.py',  # The path to the Python script
    role=role,
    framework_version='1.0-1',
    instance_count=1,
    instance_type='ml.m5.xlarge'
)

# Set the hyperparameters for XGBoost
xgboost_estimator.set_hyperparameters(max_depth=5,
                                      eta=0.2,
                                      gamma=4,
                                      min_child_weight=6,
                                      subsample=0.8,
                                      silent=0,
                                      objective='multi:softmax',
                                      num_class=3,
                                      num_round=100)



# Train -> Deploy -> Predict

In [ ]:
# Loads as buffer
test = s3_client.get_object(
    Bucket=bucket_name, 
    Key=os.path.join('data', test_name)
)
test = test['Body'].read().decode('utf-8')

# Specify the input data
data_channels = {
    'train': TrainingInput(training_data_uri, content_type='csv')
}


In [ ]:
# Train
xgboost_estimator.fit(data_channels)

# Predictor 
predictor = xgboost_estimator.deploy(initial_instance_count=1, instance_type='ml.m5.large')


response = predictor.predict(test, initial_args={"ContentType": "text/csv"})

In [ ]:
## B) predicor Batch

In [ ]:
transform_output_folder = "batch-transform-output"
output_path = "s3://{}/{}".format(sess.default_bucket(), transform_output_folder)

transformer = predictor.transformer(
    instance_count=1,
    instance_type="ml.m4.xlarge",
    output_path=output_path,
    assemble_with="Line",
    accept="csv",
)

# Delete endpoint

In [ ]:
predictor.delete_endpoint()